<a href="https://colab.research.google.com/github/ColinDonahoe119/CBB-Analytics-IGWP/blob/main/data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path
import numpy as np
import os
import sys

In [3]:
os.chdir("/content/drive/MyDrive/CBB_IGWP")
print(os.getcwd())
sys.path.append(os.getcwd())

/content/drive/MyDrive/CBB_IGWP


In [4]:
possessions = pd.read_csv("possessions.csv")
team_date_stats = pd.read_csv("team_date_stats.csv")
teams = pd.read_csv("teams.csv")

possessions.shape


(552833, 47)

In [5]:
possessions = possessions.sort_values(['gameId', 'possNum']).reset_index(drop=True)

In [7]:
## Merge possession data with date based team statistics
df = possessions.merge(
        team_date_stats[['teamId', 'gameDate', 'netRtg', 'ortg', 'drtg', 'pace']],
        on=['teamId', 'gameDate'],
        how='left'
    )
df.shape

(552833, 51)

In [8]:
# Repeat with opposing team
df = df.merge(
        team_date_stats[['teamId', 'gameDate', 'netRtg', 'ortg', 'drtg', 'pace']].rename(columns={
            'teamId': 'teamIdAgst',
            'netRtg': 'netRtgAgst',
            'ortg':   'ortgAgst',
            'drtg':   'drtgAgst',
            'pace':   'paceAgst'
        }),
        on=['teamIdAgst', 'gameDate'],
        how='left'
    )
df.shape

(552833, 55)

In [9]:
# Merge possession data with team names and markets
df = df.merge(
        teams[['teamId', 'competitionId', 'teamMarket', 'teamName']],
        on=['teamId','competitionId'],
        how='left'
    )
df.shape

(552833, 57)

In [10]:
# Repeat for opposing team
df = df.merge(
        teams[['teamId', 'competitionId', 'teamMarket', 'teamName']].rename(columns={
            'teamId':    'teamIdAgst',
            'teamMarket':'teamMarketAgst',
            'teamName':  'teamNameAgst'
        }),
        on=['teamIdAgst','competitionId'],
        how='left'
    )

In [11]:
# Eliminate teams with no netRtg(first game of the season), and keep only columns necessary for the model
df = df.dropna(subset=['netRtg', 'netRtgAgst'])
df = df[[
    'competitionId','gameId', 'possNum', 'teamId', 'teamIdAgst',
    'gameDate', 'secsInEnd', 'periodType', 'scoreEnd',
    'netRtg', 'netRtgAgst', 'ortg', 'ortgAgst', 'drtg', 'drtgAgst', 'pace', 'paceAgst',
    'teamMarket', 'teamName', 'teamMarketAgst', 'teamNameAgst'
    ]]
df.shape

(544650, 21)

In [12]:
df = df.sort_values(['gameId', 'possNum']).reset_index(drop=True)
pd.set_option('display.max_columns', None)

In [13]:
# Convert the seconds in the game to time remaining, and adjust for overtime situations
df['timeRem'] = (40 - df['secsInEnd'] / 60).round(2)
df['timeRem'] = np.where(df['timeRem'] < 0, df['timeRem'] + 5, df['timeRem'])
df['timeRem'] = np.where(df['timeRem'] < -5, df['timeRem'] + 10, df['timeRem'])
df['timeRem'] = np.where(df['timeRem'] < -10, df['timeRem'] + 15, df['timeRem'])

In [14]:
df['original_poss_teamId'] = df['teamId']

# Identify rows where teamIdAgst is the favorite (i.e., its netRtg is higher)
swap_mask = df['netRtg'] < df['netRtgAgst']

# List of column pairs to swap if teamIdAgst is the favorite
cols_to_swap = [
    ('teamId', 'teamIdAgst'),
    ('netRtg', 'netRtgAgst'),
    ('ortg', 'ortgAgst'),
    ('drtg', 'drtgAgst'),
    ('pace', 'paceAgst'),
    ('teamMarket', 'teamMarketAgst'),
    ('teamName', 'teamNameAgst')
]

# Perform the swap for identified rows
for col1, col2 in cols_to_swap:
    temp_col = df.loc[swap_mask, col1].copy()
    df.loc[swap_mask, col1] = df.loc[swap_mask, col2]
    df.loc[swap_mask, col2] = temp_col

# Create the 'poss' variable: 1 if the favorite team (now in teamId) has possession, 0 otherwise
df['poss'] = np.where(df['original_poss_teamId'] == df['teamId'], 1, 0)

# Drop the temporary original_poss_teamId column
#df = df.drop(columns=['original_poss_teamId'])

df = df.sort_values(['gameId', 'possNum']).reset_index(drop=True)



In [16]:
#Parse scoreEnd ("possessor_score-defender_score") into favorite/underdog scores
scores = df['scoreEnd'].str.split('-', expand=True).astype(int)
df['favoriteScore'] = scores[0].where(df['poss'] == 1, scores[1])
df['underdogScore']  = scores[1].where(df['poss'] == 1, scores[0])

In [18]:
# Identify net rating spread, favorite's lead, and game winner
df['spread'] = df['netRtg'] - df['netRtgAgst']
df['lead']   = df['favoriteScore'] - df['underdogScore']

df['result'] = (
  df.groupby('gameId')['favoriteScore'].transform('last') >
  df.groupby('gameId')['underdogScore'].transform('last')
).astype(int)

In [ ]:
# Split the data into Training and Testing sets
game_ids = df['gameId'].unique()
np.random.seed(42)
np.random.shuffle(game_ids)

split = int(len(game_ids) * 0.8)
train_ids = game_ids[:split]
test_ids = game_ids[split:]

train = df[df['gameId'].isin(train_ids)]
test = df[df['gameId'].isin(test_ids)]

train.to_csv(Path("/content/drive/MyDrive/CBB_IGWP") / 'train.csv', index=False)
test.to_csv(Path("/content/drive/MyDrive/CBB_IGWP") / 'test.csv', index=False)

In [ ]:
path = "/content/drive/MyDrive/CBB_IGWP/train.csv"
print(os.path.getmtime(path))  # unix timestamp
print(pd.Timestamp(os.path.getmtime(path), unit='s'))  # readable
print(pd.read_csv(path).shape)  # confirm row count looks right

1782850635.0
2026-06-30 20:17:15
(435928, 29)
